# **Semantic Web & Linked Open Data with Python**
## บทที่ 01 — Introduction

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rathachai/semantic-web/blob/main/code/ch01/ch01_notebook.ipynb)

> ## *Author* : [Rathachai CHAWUTHAI](https://rathachai.creatier.pro/) , Ph.D
> ### *Affiliation* : Computer Engineering, King Mongkut's Institute of Technology Ladkrabang (KMITL)
> #### *Updated Date* : 2026-08-23
---

<a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Creative Commons License" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/88x31.png" /></a><br />This work (course text) is licensed under a <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Creative Commons Attribution 4.0 International License</a>. Sample code cells are additionally released under the MIT License.

**References:**
*   https://rdflib.readthedocs.io/
*   https://schema.org/
*   https://www.w3.org/DesignIssues/LinkedData.html

📖 เอกสารประกอบบทนี้: [`01-introduction/`](https://github.com/Rathachai/semantic-web/tree/main/01-introduction)

---

### Setup

In [1]:
!pip install -q rdflib requests

In [2]:
import rdflib
from rdflib import Graph, URIRef, Literal, Namespace
import requests

print('rdflib', rdflib.__version__)

rdflib 7.6.0


## แนวคิดพื้นฐาน: Resource คืออะไร

ใน Semantic Web ทุกสิ่ง (คน สถานที่ แนวคิด) ถูกแทนด้วย **Resource** ที่มี URI เป็นชื่อ — ลองสร้าง Resource ตัวแรกกันดู

In [3]:
alice = URIRef("http://example.org/people/alice")

In [4]:
alice

rdflib.term.URIRef('http://example.org/people/alice')

In [5]:
name = Literal("Alice")

In [6]:
name

rdflib.term.Literal('Alice')

## Triple: หน่วยข้อมูลพื้นฐาน

ข้อความ *"Alice มีชื่อว่า Alice"* เขียนเป็น **triple** ได้ดังนี้ (subject, predicate, object)

In [7]:
from rdflib.namespace import FOAF, RDF

g = Graph()
g.add((alice, FOAF.name, name))
g.add((alice, RDF.type, FOAF.Person))

<Graph identifier=Nc31f93990850450bb3bf065c498ec272 (<class 'rdflib.graph.Graph'>)>

In [8]:
print(g.serialize(format="turtle"))

@prefix foaf: <http://xmlns.com/foaf/0.1/> .

<http://example.org/people/alice> a foaf:Person ;
    foaf:name "Alice" .




## Open World Assumption (OWA) เทียบกับ Closed World Assumption (CWA)

จุดที่ทำให้ Semantic Web ต่างจากฐานข้อมูลทั่วไปคือ **OWA** — ถ้าไม่มีข้อมูล ไม่ได้แปลว่าเป็นเท็จ แค่แปลว่า *"ยังไม่รู้"*

In [9]:
# g ไม่มี triple ที่บอกอายุของ alice เลย
ages = list(g.objects(alice, FOAF.age))
print("อายุที่พบ:", ages)
print("ภายใต้ OWA: แปลว่า 'ไม่รู้' ไม่ใช่ 'ไม่มีอายุ'")

อายุที่พบ: []
ภายใต้ OWA: แปลว่า 'ไม่รู้' ไม่ใช่ 'ไม่มีอายุ'


## ดึง Schema.org JSON-LD จากเว็บจริง

หน้าเว็บจำนวนมากฝัง structured data ในรูป **JSON-LD** ไว้ใน `<script type="application/ld+json">` — มาลองดึงจากหน้าเว็บจริงกัน

📄 โค้ดเต็ม: [`code/ch01/fetch_schema_org.py`](fetch_schema_org.py)

In [10]:
import json, re

HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; SemWebCourseBot/1.0)"}
JSONLD_PATTERN = re.compile(
    r'<script[^>]+type=["\']application/ld\+json["\'][^>]*>(.*?)</script>',
    re.DOTALL | re.IGNORECASE)

In [11]:
url = "https://schema.org/Person"
response = requests.get(url, headers=HEADERS, timeout=15)
html = response.text
print(f"ขนาด HTML: {len(html):,} characters")

ขนาด HTML: 265,945 characters


In [12]:
blocks = []
for raw in JSONLD_PATTERN.findall(html):
    try:
        blocks.append(json.loads(raw.strip()))
    except json.JSONDecodeError:
        pass

print(f"พบ JSON-LD block: {len(blocks)} รายการ")

พบ JSON-LD block: 1 รายการ


In [13]:
# ดูตัวอย่าง block แรก (ถ้ามี)
blocks[0] if blocks else "ไม่พบ JSON-LD ในหน้านี้ -- ลอง URL อื่น"

{'@context': {'brick': 'https://brickschema.org/schema/Brick#',
  'csvw': 'http://www.w3.org/ns/csvw#',
  'dc': 'http://purl.org/dc/elements/1.1/',
  'dcam': 'http://purl.org/dc/dcam/',
  'dcat': 'http://www.w3.org/ns/dcat#',
  'dcmitype': 'http://purl.org/dc/dcmitype/',
  'dcterms': 'http://purl.org/dc/terms/',
  'doap': 'http://usefulinc.com/ns/doap#',
  'foaf': 'http://xmlns.com/foaf/0.1/',
  'geo': 'http://www.opengis.net/ont/geosparql#',
  'odrl': 'http://www.w3.org/ns/odrl/2/',
  'org': 'http://www.w3.org/ns/org#',
  'owl': 'http://www.w3.org/2002/07/owl#',
  'prof': 'http://www.w3.org/ns/dx/prof/',
  'prov': 'http://www.w3.org/ns/prov#',
  'qb': 'http://purl.org/linked-data/cube#',
  'rdf': 'http://www.w3.org/1999/02/22-rdf-syntax-ns#',
  'rdfs': 'http://www.w3.org/2000/01/rdf-schema#',
  'schema': 'https://schema.org/',
  'sh': 'http://www.w3.org/ns/shacl#',
  'skos': 'http://www.w3.org/2004/02/skos/core#',
  'sosa': 'http://www.w3.org/ns/sosa/',
  'ssn': 'http://www.w3.org/ns/

## แปลง JSON-LD เป็น RDF Graph

In [14]:
web_graph = Graph()
for block in blocks:
    try:
        web_graph.parse(data=json.dumps(block), format="json-ld", publicID=url)
    except Exception as exc:
        print("แปลงไม่สำเร็จ:", exc)

print(f"จำนวน triple ที่ได้: {len(web_graph)}")

จำนวน triple ที่ได้: 624


In [15]:
preview = Graph()
for triple in list(web_graph)[:8]:
    preview.add(triple)
preview.bind("schema", "https://schema.org/")
print(preview.serialize(format="turtle"))

@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix schema: <https://schema.org/> .

schema:address schema:domainIncludes schema:Organization .

schema:agentInteractionStatistic rdfs:comment "The number of completed interactions for this entity, in a particular role (the 'agent'), in a particular action (indicated in the statistic), and in a particular context (i.e. interactionService)." .

schema:deathDate rdfs:comment "Date of death." .

schema:description owl:equivalentProperty <http://ogp.me/ns#description> .

schema:funding schema:source <https://github.com/schemaorg/schemaorg/issues/383> .

schema:image schema:domainIncludes schema:Thing .

schema:nationality schema:rangeIncludes schema:Country .

schema:owns schema:rangeIncludes schema:Thing .




## 🎯 สรุป

จาก notebook นี้จะเห็นว่า:
1. ทุกสิ่งใน Semantic Web มี **URI** เป็นชื่อ
2. ข้อมูลอยู่ในรูป **triple** (subject, predicate, object)
3. **OWA** ทำให้ "ไม่มีข้อมูล" ต่างจาก "เป็นเท็จ"
4. เว็บจริงจำนวนมากมี **JSON-LD** ฝังอยู่แล้ว พร้อมให้ดึงมาใช้ทันที

ลองเปลี่ยน `url` ในเซลล์ด้านบนเป็นเว็บอื่น (เช่นหน้าสินค้า หรือหน้าข่าว) แล้วรันดูว่าพบ JSON-LD ไหม!

---

## จบบทที่ 01 🎉

ไปต่อบทถัดไป: **บทที่ 02 — RDF**

📚 กลับหน้าคอร์ส: [https://github.com/Rathachai/semantic-web](https://github.com/Rathachai/semantic-web)

---
https://github.com/Rathachai/semantic-web

*Semantic Web & Linked Open Data with Python*